# C2 · Pré-treino do FC-JEPA em janelas contínuas

**Objetivo.** Treinar, sem rótulos, um encoder que transforma 4 s de tracking (bola + 22 jogadores)
num vetor. Depois, no repositório, uma cabeça H/A/D lê esse vetor e alimenta o mesmo Viterbi do C1.

**O que muda em relação ao FC-JEPA do laboratório**

| | FC-JEPA do lab | aqui |
|---|---|---|
| janela | uma posse (até 30 s) | 4 s contínuos, no jogo inteiro (inclui bola morta e trocas) |
| orientação | espelhada pelo time que ataca (vaza o rótulo) | a do campo; rotação de 180° sorteada |
| flag de atacante | sim | não |
| bola ausente | — | token mascarado |

**Dados (todos públicos).** 3 jogos da Metrica e 64 da PFF (Copa 2022), já na grade de 5 Hz
(`scripts/prep_jepa.py` do repo). O jogo da competição **não** entra aqui.

**Como usar**
1. *Runtime › Change runtime type › GPU*.
2. No Drive, crie `MyDrive/mtah27/caches/` e coloque os 67 `.npz` (`metrica_*.npz`, `pff_*.npz`).
3. *Runtime › Run all*. O painel da seção 7 se atualiza durante o treino.
4. Se a sessão cair, rode tudo de novo: o treino retoma da última época salva no Drive.

**Mapa do notebook:** 1 ambiente · 2 configuração · 3 tokens · 4 modelo · 5 dados · 6 métricas de
acompanhamento e linha de base · 7 treino com painel · 8 resultado e exportação · 9 como ler.


## 1. Ambiente


In [ ]:
import json, math, shutil, time
from pathlib import Path
from typing import Dict, Literal, Optional, Tuple, Union

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.functional as torch_functional
import torch.nn.init as init
from IPython.display import clear_output, display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from torch.profiler import record_function
from torch.utils.checkpoint import checkpoint
from torch.utils.data import DataLoader, Dataset, RandomSampler

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "|", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
if DEVICE == "cpu":
    print("ATENÇÃO: sem GPU. Troque o runtime antes de treinar.")

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_DIR = Path("/content/drive/MyDrive/mtah27")
except ImportError:  # fora do Colab: pasta local
    DRIVE_DIR = Path("mtah27")


## 2. Configuração

Tudo o que se ajusta está aqui. Os valores padrão são os do repositório (`configs/default.yaml`, seção `c2`).

| chave | o que é |
|---|---|
| `run_name` | nome da rodada; checkpoints e histórico ficam em `mtah27/runs/<run_name>/` |
| `T` | janela em amostras de 5 Hz (20 = 4 s) |
| `stride` | passo entre janelas possíveis (2 = 0,4 s) |
| `windows_per_epoch` | quantas janelas sorteadas por época; controla a duração da época |
| `epochs` | épocas no total (a curva de lr se ajusta a isso) |
| `d_model`, `depth`, `n_heads` | tamanho do encoder |
| `mask` | máscara do JEPA (gerador do lab). `ball_mask_prob` esconde a bola em parte das janelas, o que força o modelo a inferir onde ela está pelos jogadores |
| `n_val_pff` | jogos da PFF fora do pré-treino (perda de validação no mesmo provider do grosso do treino) |
| `val_metrica` | jogos da Metrica fora do pré-treino (perda de validação em outro provider + o fold inédito da sonda) |
| `probe_stride` | a sonda usa 1 a cada N amostras da Metrica (5 = 1 Hz) |
| `amp` | precisão mista na GPU (mais rápido). Se a perda virar `nan`, ponha `False` |
| `max_games` | `None` usa todos; um número pequeno serve para teste rápido |


In [ ]:
CFG = dict(
    run_name="c2_v2",
    T=20, stride=2, windows_per_epoch=200_000, epochs=15, batch_size=256,
    d_model=128, depth=6, n_heads=4,
    lr=1e-3, warmup_epochs=1, weight_decay=0.05, ema=(0.996, 1.0), grad_clip=10.0,
    mask=dict(mode="mixed_block", time_mask_ratio=0.8, agent_mask_ratio=0.8, num_blocks=4,
              ball_mask_prob=0.0, ball_span=0.5),
    n_val_pff=2, val_metrica=["metrica_3"], probe_stride=5,   # metrica_3: escolha fixa, feita antes de ver resultados
    amp=True, num_workers=2, plot_every=100, seed=0, max_games=None,
)
HZ = 5
PITCH_DEFAULT = (105.0, 68.0)

CACHE_SRC = DRIVE_DIR / "caches"
CACHE_DIR = Path("/content/caches") if Path("/content").exists() else CACHE_SRC
RUN_DIR = DRIVE_DIR / "runs" / CFG["run_name"]
RUN_DIR.mkdir(parents=True, exist_ok=True)
torch.manual_seed(CFG["seed"]); np.random.seed(CFG["seed"])
print("caches:", CACHE_SRC, "| rodada:", RUN_DIR)


## 3. Tokens

Cópia de `src/mtah27/tokens.py`. Cada janela vira `T × 23` tokens com 7 features: posição
(normalizada em [-1, 1] a partir do centro do campo), velocidade (m/s ÷ 10) e o tipo do agente
(bola, mandante, visitante). Agente 0 é a bola, 1–11 o mandante, 12–22 o visitante. Por janela, cada
time fica com os 11 slots mais presentes, ordenados pelo x médio. Bola ausente e slot vazio ficam
mascarados (`pad = True`) e o encoder não os vê.


In [ ]:
HOME, AWAY = 0, 1


class Tracking:
    def __init__(self, ball, players, team):
        self.ball, self.players, self.team = ball, players, team


import numpy as np


A = 23
N_PER_TEAM = 11
FEATURES = ("x", "y", "vx", "vy", "is_ball", "is_home", "is_away")
N_FEAT = len(FEATURES)
V_MAX = 12.0  # m/s; acima disso é troca de tracklet, velocidade vira 0
V_SCALE = 10.0


def velocity(pos: np.ndarray, hz: float) -> np.ndarray:
    """Diferença para trás (m/s) por frame, mesmo shape de pos [N, ..., 2]; NaN onde falta um dos dois."""
    v = np.full_like(pos, np.nan)
    v[1:] = (pos[1:] - pos[:-1]) * hz
    v[0] = v[1] if len(pos) > 1 else 0.0
    fast = np.linalg.norm(v, axis=-1) > V_MAX
    v[fast] = 0.0
    return v


def _pick_slots(players: np.ndarray, team: np.ndarray, side: int) -> np.ndarray:
    """Até 11 slots do time `side` na janela [T, S, 2]: os mais presentes, ordenados por x médio."""
    slots = np.flatnonzero(team == side)
    count = (~np.isnan(players[:, slots, 0])).sum(axis=0)
    slots, count = slots[count > 0], count[count > 0]
    top = slots[np.lexsort((slots, -count))[:N_PER_TEAM]]  # desempate pelo índice: determinístico
    mean_x = np.nanmean(players[:, top, 0], axis=0)
    return top[np.lexsort((top, mean_x))]


def window_tokens(
    trk: Tracking, vel_ball: np.ndarray, vel_players: np.ndarray, start: int, T: int, pitch
) -> tuple[np.ndarray, np.ndarray]:
    """Tokens [T * A, N_FEAT] float32 e pad [T * A] bool (True = escondido) da janela [start, start + T).

    trk já na grade (observe.subsample); vel_* de velocity() sobre o mesmo trk.
    """
    length, width = pitch
    sl = slice(start, start + T)
    players = trk.players[sl]
    pos = np.full((T, A, 2), np.nan, np.float32)
    vel = np.zeros((T, A, 2), np.float32)
    flags = np.zeros((A, 3), np.float32)
    pos[:, 0], vel[:, 0], flags[0, 0] = trk.ball[sl], vel_ball[sl], 1.0
    for side, first, col in ((HOME, 1, 1), (AWAY, 1 + N_PER_TEAM, 2)):
        chosen = _pick_slots(players, trk.team, side)
        k = len(chosen)
        pos[:, first : first + k] = players[:, chosen]
        vel[:, first : first + k] = vel_players[sl][:, chosen]
        flags[first : first + N_PER_TEAM, col] = 1.0
    pad = np.isnan(pos[..., 0])
    xy = (pos - np.array([length / 2, width / 2], np.float32)) / np.array([length / 2, width / 2], np.float32)
    vel = np.nan_to_num(vel) / V_SCALE
    tok = np.concatenate([np.nan_to_num(xy), vel, np.broadcast_to(flags, (T, A, 3))], axis=-1)
    tok[pad] = 0.0
    return tok.reshape(T * A, N_FEAT).astype(np.float32), pad.reshape(T * A)


def flip_tokens(tok: np.ndarray) -> np.ndarray:
    """Rotação de 180° do campo (x, y, vx, vy trocam de sinal); funciona em numpy e torch."""
    out = tok.clone() if hasattr(tok, "clone") else tok.copy()
    out[..., :4] = -out[..., :4]
    return out


## 4. Modelo

Cópia dos 5 arquivos do FC-JEPA (laboratório, commit 334b3a3); só os imports mudaram. Resumo: o
*context encoder* vê os tokens visíveis, o *predictor* tenta prever a representação dos tokens
escondidos, e o *target encoder* (média móvel do context encoder, sem gradiente) dá o alvo. A perda é
L1 entre previsão e alvo em três níveis: token, agente e posse (`token_loss`, `agent_loss`,
`poss_loss`).


In [ ]:
# ---- common.py (fc-jepa, commit 334b3a3) ----
# backbone/common.py

import math
from typing import Dict, Optional, Tuple, Union

import torch
import torch.nn as nn
import torch.nn.functional as F


class SinusoidalPE(nn.Module):
    """
    Classic transformer PE. We build separate PEs for time (T) and agent slots (A),
    then sum them after broadcasting to tokens.
    """

    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim
        assert dim % 2 == 0, "PE dim must be even"

    def _pe_1d(self, n: int, device):
        d = self.dim
        pos = torch.arange(n, device=device).float().unsqueeze(1)  # [n,1]
        i = torch.arange(0, d, 2, device=device).float()  # [d/2]
        div = torch.exp(-i * math.log(10000.0) / d)
        pe = torch.zeros(n, d, device=device)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        return pe  # [n, d]

    def forward(self, T: int, A: int, device=None):
        pe_t = self._pe_1d(T, device)  # [T, d]
        pe_a = self._pe_1d(A, device)  # [A, d]
        return pe_t, pe_a


# ---- Agent-slot embeddings ----


class AgentSlotEmbedding(nn.Module):
    """
    Learnable embeddings per agent slot id (0..A-1). No team IDs.
    """

    def __init__(self, A: int, dim: int):
        super().__init__()
        self.emb = nn.Embedding(A, dim)

    def forward(self, A: int, device=None):
        idx = torch.arange(A, device=device)
        return self.emb(idx)  # [A, dim]


def build_same_agent_bias(S: int, A: int, device, bias_value: float) -> torch.Tensor:
    """
    Legacy helper (O(S^2)): Return [S,S] float bias matrix: +bias where tokens share the same agent slot.
    Kept for backward compatibility, but DO NOT USE in training paths if you care about OOM.
    """
    idx = torch.arange(S, device=device)
    a_i = (idx % A).unsqueeze(1)  # [S,1]
    a_j = (idx % A).unsqueeze(0)  # [1,S]
    same = (a_i == a_j).float()
    return same * bias_value  # [S,S]


# -----------------------------------------------------------------------------
# Efficient same-agent bias implementation
# -----------------------------------------------------------------------------
_ONE_HOT_CACHE: Dict[Tuple[str, str, int, int, int], torch.Tensor] = {}
# key: (device_str, dtype_str, length, A, A_pad) -> [1,1,length,A_pad]


def _round_up_to_multiple(x: int, m: int) -> int:
    if m <= 1:
        return x
    return ((x + m - 1) // m) * m


def _get_cached_one_hot(length: int, A: int, A_pad: int, device, dtype) -> torch.Tensor:
    """
    Returns [1,1,length,A_pad] float tensor where row i is one-hot(agent_id=i%A) in dim A_pad.
    Caches per (device,dtype,length,A,A_pad). This avoids repeatedly rebuilding one-hots each layer/step.
    """
    device_str = str(device)
    dtype_str = str(dtype)
    key = (device_str, dtype_str, int(length), int(A), int(A_pad))

    t = _ONE_HOT_CACHE.get(key, None)
    if t is not None:
        return t

    # Simple cache control to avoid unbounded growth
    if len(_ONE_HOT_CACHE) > 64:
        _ONE_HOT_CACHE.clear()

    # Build one-hot via eye lookup (fast, deterministic)
    # agent_ids: [length]
    agent_ids = torch.arange(length, device=device) % int(A)
    eye = torch.eye(int(A_pad), device=device, dtype=dtype)
    one_hot = eye.index_select(0, agent_ids)  # [length, A_pad]
    one_hot = one_hot.unsqueeze(0).unsqueeze(0).contiguous()  # [1,1,length,A_pad]

    _ONE_HOT_CACHE[key] = one_hot
    return one_hot


def _bias_to_alpha_and_sign(
    bias_value: Union[float, torch.Tensor], d0: int, device, dtype
) -> Tuple[Union[float, torch.Tensor], float]:
    """
    We want to add a constant bias 'b' to attention logits for same-agent pairs.

    In the fast SDPA path we do it by augmenting q/k:
      logits = (q_aug @ k_aug^T) * (1/sqrt(d0))
    and choose alpha such that:
      (alpha^2 / sqrt(d0)) == |b|   -> alpha = sqrt(|b| * sqrt(d0))

    Returns:
      alpha: float or scalar tensor (keeps grad if bias_value is a tensor param)
      sign:  +1.0 or -1.0 (detached; if bias_value is tensor and flips sign, sign gradient is undefined anyway)
    """
    if torch.is_tensor(bias_value):
        # Keep gradient through magnitude; sign is treated as constant (detached)
        sign = 1.0 if float(bias_value.detach().item()) >= 0.0 else -1.0
        bias_abs = torch.abs(bias_value).to(device=device, dtype=dtype)
        alpha = torch.sqrt(bias_abs * math.sqrt(float(d0)))
        return alpha, sign

    # float path
    b = float(bias_value)
    if b == 0.0:
        return 0.0, 1.0
    sign = 1.0 if b >= 0.0 else -1.0
    alpha = math.sqrt(abs(b) * math.sqrt(float(d0)))
    return alpha, sign


def apply_sdpa(q, k, v, attn_mask_bool: Optional[torch.Tensor], add_bias):
    """
    q: [B, H, S_q, d_h]
    k,v: [B, H, S_k, d_h]
    attn_mask_bool: [B, S_q, S_k] or [B,1,S_q,S_k] or [B,H,S_q,S_k] or
                    key-padding broadcast [B,1,1,S_k] (True=BLOCK) or None
    add_bias:
      - None
      - ("same_agent", A, bias_value)   # efficient, no [S,S]
      - (legacy) Tensor [S,S] or [B,S,S] or [B,1,S,S]/[B,H,S,S]
    """
    B, H, S_q, d0 = q.shape
    if k.dim() != 4 or v.dim() != 4:
        raise ValueError(f"k/v must be 4D [B,H,S_k,d], got k={k.dim()} v={v.dim()}")
    if k.shape[:2] != (B, H) or v.shape[:2] != (B, H):
        raise ValueError(
            f"k/v batch/head mismatch: q={tuple(q.shape)} k={tuple(k.shape)} v={tuple(v.shape)}"
        )
    S_k = k.shape[2]
    if v.shape[2] != S_k:
        raise ValueError(f"v must have same S_k as k. got k={S_k}, v={v.shape[2]}")

    # Normalize bool mask shape to [B,1,*,S_k] (broadcast over heads) if present
    if attn_mask_bool is not None:
        if attn_mask_bool.dim() == 3:
            attn_mask_bool = attn_mask_bool.unsqueeze(1)  # [B,1,L,S_k]
        elif attn_mask_bool.dim() == 4:
            pass
        else:
            raise ValueError(f"attn_mask_bool must have dim 3 or 4, got {attn_mask_bool.dim()}")

        if attn_mask_bool.size(-1) != S_k:
            raise ValueError(
                f"attn_mask_bool last dim must be S_k={S_k}, got {attn_mask_bool.size(-1)}"
            )

        L = attn_mask_bool.size(-2)  # query length

        # L==1 is broadcastable key padding mask: DO NOT expand it.
        if L == 1:
            pass
        elif L == S_q:
            pass
        else:
            raise ValueError(
                f"attn_mask_bool has query length L={L}, but q length is S_q={S_q}. "
                "Fix mask construction upstream (make L==1 or L==S_q)."
            )

    def _needs_chunked(mask: Optional[torch.Tensor], S_q: int, S_k: int) -> bool:
        if mask is None:
            return False
        if mask.dim() != 4:
            return True
        L = mask.size(-2)
        if L == 1:
            return False
        # Only chunk for genuinely large per-query masks
        return S_q >= 2048 or S_k >= 2048

    # -------------------------------------------------------------------------
    # Fast SDPA path
    # -------------------------------------------------------------------------
    if not _needs_chunked(attn_mask_bool, S_q, S_k):
        # aqui True = BLOQUEIA; no SDPA do torch, máscara booleana True = PODE ATENDER. Sem inverter,
        # o encoder atendia só aos tokens escondidos (alvos e slots vazios) e nunca aos visíveis.
        if attn_mask_bool is not None:
            attn_mask_bool = ~attn_mask_bool
        if add_bias is None:
            return F.scaled_dot_product_attention(
                q, k, v, attn_mask=attn_mask_bool, is_causal=False
            )

        if isinstance(add_bias, tuple) and len(add_bias) == 3 and add_bias[0] == "same_agent":
            _, A, bias_value = add_bias
            if A is None:
                return F.scaled_dot_product_attention(
                    q, k, v, attn_mask=attn_mask_bool, is_causal=False
                )

            # If bias is effectively zero
            if (torch.is_tensor(bias_value) and float(bias_value.detach().item()) == 0.0) or (
                (not torch.is_tensor(bias_value)) and float(bias_value) == 0.0
            ):
                return F.scaled_dot_product_attention(
                    q, k, v, attn_mask=attn_mask_bool, is_causal=False
                )

            # -----------------------------------------------------------------
            # SAME-AGENT BIAS via q/k augmentation, but with A padded to multiple of 8
            # to keep head dim aligned (helps flash/mem-efficient kernels).
            #
            # We preserve original scaling based on d0 by setting SDPA scale=1/sqrt(d0).
            # -----------------------------------------------------------------
            scale = 1.0 / math.sqrt(float(d0))

            A_int = int(A)
            # pad to 8 to avoid awkward head dims (e.g., 64+23=87); 64+24=88 is 8-aligned
            A_pad = _round_up_to_multiple(A_int, 8)

            alpha, sign = _bias_to_alpha_and_sign(bias_value, d0=d0, device=q.device, dtype=q.dtype)

            one_hot_q = _get_cached_one_hot(S_q, A_int, A_pad, device=q.device, dtype=q.dtype)
            one_hot_k = _get_cached_one_hot(S_k, A_int, A_pad, device=q.device, dtype=q.dtype)
            # [1,1,S_q,A_pad], [1,1,S_k,A_pad]

            # Expand without allocating per-(B,H) copies; concatenation allocates final q_aug/k_aug
            q_extra = (alpha * one_hot_q).expand(B, H, S_q, A_pad)
            k_extra = (sign * alpha * one_hot_k).expand(B, H, S_k, A_pad)

            q_aug = torch.cat([q, q_extra], dim=-1)
            k_aug = torch.cat([k, k_extra], dim=-1)

            return F.scaled_dot_product_attention(
                q_aug, k_aug, v, attn_mask=attn_mask_bool, is_causal=False, scale=scale
            )

        # Legacy tensor bias handling (discouraged with per-example masks)
        if torch.is_tensor(add_bias):
            if attn_mask_bool is None and add_bias.dim() == 2:
                return F.scaled_dot_product_attention(q, k, v, attn_mask=add_bias, is_causal=False)
            raise RuntimeError(
                "apply_sdpa: tensor add_bias with per-example bool mask can allocate huge buffers. "
                "Use add_bias=('same_agent', A, bias_value) or rely on chunked fallback."
            )

        raise TypeError(f"apply_sdpa: unsupported add_bias type: {type(add_bias)}")

    # -------------------------------------------------------------------------
    # Chunked attention fallback (mask-friendly, avoids [B,H,S,S])
    # -------------------------------------------------------------------------
    q_dtype = q.dtype
    device = q.device

    # Stable softmax: do matmul in fp32
    qf = q.float()
    kf = k.float()
    vf = v.float()

    scale = 1.0 / math.sqrt(float(d0))

    # Precompute agent ids for same-agent bias if requested
    same_agent = False
    bias_value_f = 0.0
    A_bias = None
    if isinstance(add_bias, tuple) and len(add_bias) == 3 and add_bias[0] == "same_agent":
        _, A_bias, bias_value = add_bias
        if A_bias is not None:
            A_bias = int(A_bias)
            if torch.is_tensor(bias_value):
                bias_value_f = float(bias_value.detach().item())
            else:
                bias_value_f = float(bias_value)
            same_agent = bias_value_f != 0.0

            if same_agent:
                agent_k = torch.arange(S_k, device=device) % A_bias  # [S_k]

    blk = 256
    out = torch.empty((B, H, S_q, d0), device=device, dtype=q_dtype)

    for qs in range(0, S_q, blk):
        qe = min(S_q, qs + blk)
        q_blk = qf[:, :, qs:qe, :]  # [B,H,blk,d]
        scores = torch.matmul(q_blk, kf.transpose(-2, -1)) * scale  # [B,H,blk,S_k] in fp32

        if same_agent:
            agent_q = torch.arange(qs, qe, device=device) % A_bias
            same = (agent_q[:, None] == agent_k[None, :]).to(dtype=scores.dtype)
            scores = scores + (bias_value_f * same)[None, None, :, :]

        if torch.is_tensor(add_bias):
            if add_bias.dim() == 2:
                scores = scores + add_bias[qs:qe, :].to(scores.dtype)[None, None, :, :]
            elif add_bias.dim() == 3:
                scores = scores + add_bias[:, qs:qe, :].to(scores.dtype)[:, None, :, :]
            elif add_bias.dim() == 4:
                scores = scores + add_bias[:, :, qs:qe, :].to(scores.dtype)
            else:
                raise ValueError(f"add_bias tensor dim must be 2/3/4, got {add_bias.dim()}")

        if attn_mask_bool is not None:
            m_slice = attn_mask_bool[:, :, qs:qe, :]  # [B,1,blk,S_k] or [B,H,blk,S_k]
            all_masked = m_slice.all(dim=-1, keepdim=True)
            scores = scores.masked_fill(m_slice, float("-inf"))
            scores = torch.where(all_masked, torch.zeros_like(scores), scores)

        attn = torch.softmax(scores, dim=-1, dtype=torch.float32).to(q_dtype)  # [B,H,blk,S_k]
        out[:, :, qs:qe, :] = torch.matmul(attn, vf)  # matmul in fp32->bf16 path is ok

    return out


class MLP(nn.Module):
    def __init__(self, d_model: int, mult: int = 4, p: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, mult * d_model),
            nn.GELU(),
            nn.Dropout(p),
            nn.Linear(mult * d_model, d_model),
        )

    def forward(self, x):
        return self.net(x)


# ---- pooling.py (fc-jepa, commit 334b3a3) ----
# backbone/pooling.py

import torch
import torch.nn as nn


class TemporalPool(nn.Module):
    """
    Per-agent temporal pooling over T: mean|max|attn.
    Input:  H_token [B, T, A, D]
    Output: H_agent [B, A, D]
    """

    def __init__(self, d_model: int, mode: str = "attn"):
        super().__init__()
        self.mode = mode
        if mode == "attn":
            self.query = nn.Parameter(torch.randn(1, 1, d_model))  # [1,1,D]
            self.proj = nn.Linear(d_model, d_model)

    def forward(self, H_token: torch.Tensor, presence: torch.Tensor):
        # presence: [B,T,A] (True where token exists); use as mask in pooling
        B, T, A, D = H_token.shape
        x = H_token  # [B,T,A,D]
        mask = ~presence  # True where pad
        if self.mode == "mean":
            x = x.masked_fill(mask[..., None], 0.0)
            denom = (~mask).sum(dim=1).clamp(min=1)[..., None]  # [B,A,1]
            return x.sum(dim=1) / denom  # [B,A,D]
        elif self.mode == "max":
            x = x.masked_fill(mask[..., None], float("-inf"))
            return x.max(dim=1).values  # [B,A,D]
        else:  # attn
            # Learned query per agent
            q = self.query.expand(B, A, -1)  # [B,A,D]
            k = self.proj(x)  # [B,T,A,D]
            attn = torch.einsum("bad,btad->bta", q, k) / (D**0.5)  # [B,T,A]
            # mask True -> invalid positions
            attn = attn.masked_fill(mask, float("-inf"))

            # Safe masked softmax over time dim. If an agent has no valid tokens
            # we force its attention weights to zero to avoid NaNs.
            # Compute per-agent max (for numerical stability)
            attn_max = attn.max(dim=1, keepdim=True).values  # [B,1,A]
            # Replace -inf (when all positions masked) with 0 to avoid NaNs
            attn_max = torch.where(torch.isinf(attn_max), torch.zeros_like(attn_max), attn_max)
            exp_attn = torch.exp(attn - attn_max)
            exp_attn = exp_attn.masked_fill(mask, 0.0)
            denom = exp_attn.sum(dim=1, keepdim=True).clamp_min(1.0)  # avoid div0
            w = (exp_attn / denom)[..., None]  # [B,T,A,1]

            H_agent = (w * x).sum(dim=1)  # [B,A,D]

            # Zero-out agents with no presence
            agent_is_present = presence.any(dim=1)  # [B,A]
            H_agent = H_agent.masked_fill(~agent_is_present[..., None], 0.0)
            return H_agent


class PossessionPool(nn.Module):
    """
    Attentive pooling over all tokens to get a possession embedding.
    Input:  H_token [B, T, A, D], presence [B,T,A]
    Output: h_poss [B, D]
    """

    def __init__(self, d_model: int):
        super().__init__()
        self.query = nn.Parameter(torch.randn(1, d_model))
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, H_token: torch.Tensor, presence: torch.Tensor):
        B, T, A, D = H_token.shape
        q = self.query.expand(B, -1)  # [B,D]
        k = self.proj(H_token)  # [B,T,A,D]
        logits = torch.einsum("bd,btad->bta", q, k) / (D**0.5)  # [B,T,A]
        mask = ~presence
        logits = logits.masked_fill(mask, float("-inf"))

        # Safe masked softmax over flattened (T*A). If all positions are masked
        # for a sample, we return a zero vector for the possession embedding.
        flat = logits.view(B, -1)  # [B, T*A]
        flat_max = flat.max(dim=1, keepdim=True).values  # [B,1]
        flat_max = torch.where(torch.isinf(flat_max), torch.zeros_like(flat_max), flat_max)
        exp_flat = torch.exp(flat - flat_max)
        # Zero-out masked positions
        exp_flat = exp_flat * (~mask).view(B, -1).float()
        denom = exp_flat.sum(dim=1, keepdim=True)
        # If denom==0 -> all masked -> return zeros
        zero_den = denom.squeeze(1) == 0
        denom = denom.clamp_min(1.0)
        w_flat = exp_flat / denom  # [B, T*A]
        w = w_flat.view(B, T, A, 1)
        h = (w * H_token).sum(dim=(1, 2))  # [B,D]
        # Zero-out any samples that had no valid tokens
        if zero_den.any():
            h = h.masked_fill(zero_den.unsqueeze(1), 0.0)
        return h


# ---- backbone_joint.py (fc-jepa, commit 334b3a3) ----
# backbone/backbone_joint.py

import torch
import torch.nn as nn
from torch.profiler import record_function
from torch.utils.checkpoint import checkpoint  # Import checkpoint utility



class JointBlock(nn.Module):
    def __init__(
        self, d_model: int, n_heads: int, A: int, same_agent_bias=False, bias_init=0.2, p=0.1
    ):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.Wq = nn.Linear(d_model, d_model, bias=True)
        self.Wk = nn.Linear(d_model, d_model, bias=True)
        self.Wv = nn.Linear(d_model, d_model, bias=True)
        self.Wo = nn.Linear(d_model, d_model, bias=True)
        self.mlp = MLP(d_model, p=p)
        self.n_heads = n_heads
        self.A = A
        self.same_agent_bias = same_agent_bias
        self.bias_strength = nn.Parameter(torch.tensor(float(bias_init)))

    def forward(self, x, attn_mask_bool, key_pad_bool, T, A):
        """
        x: [B, S, D], attn_mask_bool/key_pad_bool: [B,S,S]/[B,S]
        """
        B, S, D = x.shape
        with record_function("jointblock_norm_qkv"):
            h = self.norm1(x)
            q = self.Wq(h)
            k = self.Wk(h)
            v = self.Wv(h)  # [B,S,D]
            H = self.n_heads
            d_h = D // H
            q = q.view(B, S, H, d_h).transpose(1, 2)  # [B,H,S,d_h]
            k = k.view(B, S, H, d_h).transpose(1, 2)
            v = v.view(B, S, H, d_h).transpose(1, 2)

        # Masks (avoid materializing [B,H,S,S])
        with record_function("jointblock_mask_bias"):
            merged_mask = None
            if key_pad_bool is not None:
                key_mask = key_pad_bool[:, None, None, :]  # [B,1,1,S]
                merged_mask = key_mask if merged_mask is None else (merged_mask | key_mask)
            if attn_mask_bool is not None:
                if attn_mask_bool.dim() == 3:  # [B,S,S] -> [B,1,S,S]
                    attn_mask_bool = attn_mask_bool.unsqueeze(1)
                merged_mask = (
                    attn_mask_bool if merged_mask is None else (merged_mask | attn_mask_bool)
                )

            add_bias = None
            if self.same_agent_bias:
                add_bias = ("same_agent", int(A), self.bias_strength)

        with record_function("jointblock_sdpa"):
            out = apply_sdpa(q, k, v, merged_mask, add_bias)

        with record_function("jointblock_out_mlp"):
            out = out.transpose(1, 2).reshape(B, S, D)
            x = x + self.Wo(out)
            x = x + self.mlp(self.norm2(x))
        return x


class JointBackbone(nn.Module):
    """
    Joint space-time attention over S=T*A tokens.
    """

    def __init__(
        self,
        in_dim: int,
        d_model: int,
        depth: int,
        n_heads: int,
        A: int = 23,
        time_pe_dim: int = 64,
        agent_emb_dim: int = 64,
        same_agent_bias: bool = False,
        p: float = 0.1,
        pool_temporal: str = "attn",
    ):
        super().__init__()
        self.A = A
        self.proj_in = nn.Sequential(
            nn.Linear(in_dim, d_model), nn.ReLU(), nn.Linear(d_model, d_model)
        )
        self.time_pe = SinusoidalPE(time_pe_dim)
        self.agent_emb = AgentSlotEmbedding(A, agent_emb_dim)
        self.pe_proj = nn.Linear(time_pe_dim + agent_emb_dim, d_model)
        self.blocks = nn.ModuleList(
            [
                JointBlock(d_model, n_heads, A, same_agent_bias=same_agent_bias, p=p)
                for _ in range(depth)
            ]
        )
        self.norm = nn.LayerNorm(d_model)
        self.temp_pool = TemporalPool(d_model, mode=pool_temporal)
        self.poss_pool = PossessionPool(d_model)

    def forward(self, tokens, attn_mask_bool, key_pad_bool, T, A, use_checkpoint=False):
        """
        tokens: [B, S, F], S=T*A
        attn_mask_bool: optional; may be [B,S,S] or already broadcastable for SDPA.
        key_pad_bool: [B,S] (True=pad) — converted to [B,1,1,S] inside the block.
        Returns:
            H_token [B,T,A,D], H_agent [B,A,D], h_poss [B,D]
        """
        B, S, F = tokens.shape
        assert S == T * A == T * self.A
        D = self.proj_in[2].out_features
        with record_function("proj_in"):
            x = self.proj_in(tokens)  # [B,S,D]

        # Add time & agent encodings
        with record_function("positional_encoding"):
            pe_t, pe_a = self.time_pe(T, A, device=tokens.device)  # [T,dt], [A,da]
            pe = torch.cat(
                [
                    pe_t.repeat_interleave(A, dim=0),  # [T,dt] -> [S,dt]
                    pe_a.repeat(T, 1),  # [A,da] -> [S,da]
                ],
                dim=1,
            )  # [S, dt+da]
            pe = self.pe_proj(pe).unsqueeze(0).expand(B, -1, -1)  # [B,S,D]
            x = x + pe

        for blk in self.blocks:
            if use_checkpoint and x.requires_grad:
                x = checkpoint(blk, x, attn_mask_bool, key_pad_bool, T, A, use_reentrant=False)
            else:
                x = blk(x, attn_mask_bool, key_pad_bool, T=T, A=A)

        with record_function("norm_pool"):
            x = self.norm(x)  # [B,S,D]
            H_token = x.view(B, T, A, D)
            presence = (~key_pad_bool).view(B, T, A)
            H_agent = self.temp_pool(H_token, presence)  # [B,A,D]
            h_poss = self.poss_pool(H_token, presence)  # [B,D]
        return H_token, H_agent, h_poss


# ---- masking.py (fc-jepa, commit 334b3a3) ----
# jepa/masking.py

from typing import Literal

import torch


def generate_mask_vjepa(
    T: int,
    A: int,
    batch_size: int,
    device: torch.device,
    mask_mode: Literal["time_span", "agent_slots", "mixed_block"],
    time_mask_ratio: float = 0.3,
    agent_mask_ratio: float = 0.3,
    num_blocks: int = 4,
) -> torch.Tensor:
    """
    Generates a boolean mask [B, T, A] (True = MASKED/TARGET, False = VISIBLE/CONTEXT).
    """
    mask = torch.zeros((batch_size, T, A), dtype=torch.bool, device=device)

    if mask_mode == "time_span":
        mask_len = int(T * time_mask_ratio)
        if mask_len == 0:
            mask_len = 1

        max_start = T - mask_len
        start_t = torch.randint(0, max(1, max_start + 1), (batch_size,), device=device)

        for i in range(batch_size):
            end_t = min(T, start_t[i].item() + mask_len)
            mask[i, start_t[i].item() : end_t, :] = True

    elif mask_mode == "agent_slots":
        mask_count = int(A * agent_mask_ratio)
        if mask_count == 0:
            mask_count = 1

        for i in range(batch_size):
            masked_agents = torch.randperm(A, device=device)[:mask_count]
            mask[i, :, masked_agents] = True

    elif mask_mode == "mixed_block":
        mask_A_count = max(1, int(A * agent_mask_ratio))
        mask_T_len = max(1, int(T * (time_mask_ratio * 0.5)))

        for i in range(batch_size):
            for _ in range(max(1, num_blocks)):
                if mask_A_count >= A:
                    start_a = 0
                    end_a = A
                else:
                    max_start_a = A - mask_A_count
                    start_a = torch.randint(0, max_start_a + 1, (1,), device=device).item()
                    end_a = start_a + mask_A_count

                if mask_T_len >= T:
                    start_t = 0
                    end_t = T
                else:
                    max_start_t = T - mask_T_len
                    start_t = torch.randint(0, max_start_t + 1, (1,), device=device).item()
                    end_t = start_t + mask_T_len

                mask[i, start_t:end_t, start_a:end_a] = True

    # Safety: avoid all/none masking per sample (generic, not pad-aware)
    Btot = mask.shape[0]
    for i in range(Btot):
        if not mask[i].any():
            t = torch.randint(0, T, (1,), device=device).item()
            a = torch.randint(0, A, (1,), device=device).item()
            mask[i, t, a] = True
        elif mask[i].all():
            t = torch.randint(0, T, (1,), device=device).item()
            a = torch.randint(0, A, (1,), device=device).item()
            mask[i, t, a] = False

    return mask


# ---- model.py (fc-jepa, commit 334b3a3) ----
# jepa/model.py

from typing import Literal, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as torch_functional
import torch.nn.init as init
from torch.profiler import record_function

MAX_N_AGENTS = 23


class TransformerPredictor(nn.Module):
    """
    Transformer encoder predictor over [poss_token + agent_tokens].
    """

    def __init__(
        self,
        d_model: int,
        A: int = MAX_N_AGENTS,
        n_layers: int = 4,
        n_heads: int = 4,
        p: float = 0.1,
    ):
        super().__init__()
        self.A = A
        self.d_model = d_model

        self.poss_token = nn.Parameter(torch.randn(1, 1, d_model))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4, dropout=p, batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.norm = nn.LayerNorm(d_model)

        init.trunc_normal_(self.poss_token, std=0.02)

    def forward(
        self, H_agent_ctx: torch.Tensor, h_poss_ctx: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        H_agent_ctx: [B, A, D]
        h_poss_ctx : [B, D]
        """
        B, A, D = H_agent_ctx.shape

        poss_token_bias = self.poss_token.expand(B, 1, D)
        poss_token_input = poss_token_bias + h_poss_ctx.unsqueeze(1)  # [B,1,D]
        sequence = torch.cat([poss_token_input, H_agent_ctx], dim=1)  # [B,A+1,D]

        encoded_sequence = self.transformer_encoder(sequence)  # [B,A+1,D]

        h_poss_pred = self.norm(encoded_sequence[:, 0, :])  # [B,D]
        H_agent_pred = self.norm(encoded_sequence[:, 1:, :])  # [B,A,D]
        return H_agent_pred, h_poss_pred


class FCJEPAModel(nn.Module):
    """
    V-JEPA-style pretraining over soccer possession tokens S=T*A.

    Inputs:
      x_full       : [B, S, F], where S = T * A
      key_pad_full : [B, S], True means PAD (ignored / hidden)
    """

    def __init__(
        self,
        in_dim_enc: int,
        d_model: int,
        depth: int,
        n_heads: int,
        same_agent_bias: bool,
        A: int = MAX_N_AGENTS,
    ):
        super().__init__()
        self.A = A
        self.d_model = d_model

        self.context_encoder = JointBackbone(
            in_dim=in_dim_enc,
            d_model=d_model,
            depth=depth,
            n_heads=n_heads,
            A=A,
            same_agent_bias=same_agent_bias,
        )

        self.target_encoder = JointBackbone(
            in_dim=in_dim_enc,
            d_model=d_model,
            depth=depth,
            n_heads=n_heads,
            A=A,
            same_agent_bias=same_agent_bias,
        )

        self.predictor = TransformerPredictor(d_model=d_model, A=A, n_layers=4, n_heads=n_heads)
        self.token_predictor = MLP(d_model, p=0.1)

        self._init_weights()

        self.target_encoder.load_state_dict(self.context_encoder.state_dict())
        for p in self.target_encoder.parameters():
            p.requires_grad = False

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                init.kaiming_uniform_(m.weight, mode="fan_in", nonlinearity="relu")
                if m.bias is not None:
                    init.constant_(m.bias, 0)
            elif isinstance(m, nn.LayerNorm):
                init.constant_(m.weight, 1.0)
                init.constant_(m.bias, 0.0)
            elif isinstance(m, nn.Embedding):
                init.trunc_normal_(m.weight, mean=0.0, std=0.02)

    def get_mask_modes(self):
        return Literal["time_span", "agent_slots", "mixed_block"]

    @staticmethod
    def _ensure_nonpad_mask_sane(mask_bta: torch.Tensor, pad_bta: torch.Tensor) -> torch.Tensor:
        """
        Restrict target mask to non-pad tokens and ensure:
          - at least one non-pad target token
          - at least one non-pad context token
        """
        device = mask_bta.device
        B, T, A = mask_bta.shape

        nonpad = ~pad_bta  # True where token is real
        mask_bta = mask_bta & nonpad

        # Per-sample corrections
        for i in range(B):
            nonpad_i = nonpad[i]  # [T,A]
            if not nonpad_i.any():
                # All padding: leave mask empty; caller should skip sample.
                continue

            # If no targets in nonpad region, pick one nonpad token to mask
            if not mask_bta[i].any():
                idx = torch.nonzero(nonpad_i, as_tuple=False)
                pick = idx[torch.randint(0, idx.size(0), (1,), device=device).item()]
                mask_bta[i, pick[0].item(), pick[1].item()] = True

            # If everything nonpad is masked, unmask one nonpad token
            if (mask_bta[i] == nonpad_i).all():
                idx = torch.nonzero(nonpad_i, as_tuple=False)
                pick = idx[torch.randint(0, idx.size(0), (1,), device=device).item()]
                mask_bta[i, pick[0].item(), pick[1].item()] = False

        return mask_bta

    def forward(
        self,
        x_full: torch.Tensor,
        key_pad_full: torch.Tensor,
        T: int,
        A: int,
        mask_mode: str,
        mask_kwargs: dict,
        use_checkpoint: bool = False,
    ):
        """
        x_full       : [B, S, F] with S = T * A
        key_pad_full : [B, S] (True = pad)

        Returns:
          total_loss, token_loss, agent_loss, poss_loss, H_token_pred, H_agent_pred, h_poss_pred
        """
        B, S, _F = x_full.shape
        device = x_full.device

        if S != T * A:
            raise ValueError(f"[JEPA] S != T*A (S={S}, T={T}, A={A})")

        real_pad_s = key_pad_full.bool()  # [B,S]
        pad_bta = real_pad_s.view(B, T, A)  # [B,T,A]

        # 1) Sample mask (True = target) then restrict to non-pad region
        with record_function("mask_generate+sanitize"):
            mask_bta = generate_mask_vjepa(
                T=T, A=A, batch_size=B, device=device, mask_mode=mask_mode, **mask_kwargs
            )
            mask_bta = self._ensure_nonpad_mask_sane(mask_bta, pad_bta)
            mask_sa = mask_bta.view(B, S)  # [B,S]

        # If any sample is fully padded, caller should skip; but keep safe behavior
        if real_pad_s.all(dim=1).any():
            # loss will be zero; training loop should skip these samples
            pass

        # Context sees ONLY context tokens: non-pad AND not masked
        key_pad_ctx = real_pad_s | mask_sa  # True = hidden from context

        # Target sees ONLY target tokens: non-pad AND masked
        key_pad_tgt = real_pad_s | (~mask_sa)

        # 2) Context encoder
        with record_function("context_encoder"):
            H_token_ctx, H_agent_ctx, h_poss_ctx = self.context_encoder(
                tokens=x_full,
                attn_mask_bool=None,
                key_pad_bool=key_pad_ctx,
                T=T,
                A=A,
                use_checkpoint=use_checkpoint,
            )

        # 3) Predictor
        with record_function("predictor_transformer"):
            H_agent_pred, h_poss_pred = self.predictor(H_agent_ctx, h_poss_ctx)
        with record_function("predictor_token_mlp"):
            H_token_pred = self.token_predictor(H_token_ctx)

        # 4) Target encoder (EMA)
        with record_function("target_encoder_ema"):

            with torch.no_grad():
                H_token_tgt, H_agent_tgt, h_poss_tgt = self.target_encoder(
                    tokens=x_full,
                    attn_mask_bool=None,
                    key_pad_bool=key_pad_tgt,
                    T=T,
                    A=A,
                    use_checkpoint=False,
                )

        # 5) Loss on TARGET region only (and non-pad)
        with record_function("loss_token_agent_poss"):
            token_mask = mask_bta & (~pad_bta)  # [B,T,A]

            if token_mask.any():
                H_token_pred_masked = H_token_pred[token_mask]  # [N,D]
                H_token_tgt_masked = H_token_tgt[token_mask]
                token_loss = torch_functional.l1_loss(
                    H_token_pred_masked, H_token_tgt_masked, reduction="mean"
                )
            else:
                token_loss = torch.zeros((), device=device)

            agent_has_target = token_mask.any(dim=1)  # [B,A]
            if agent_has_target.any():
                H_agent_pred_masked = H_agent_pred[agent_has_target]  # [N,D]
                H_agent_tgt_masked = H_agent_tgt[agent_has_target]
                agent_loss = torch_functional.l1_loss(
                    H_agent_pred_masked, H_agent_tgt_masked, reduction="mean"
                )
            else:
                agent_loss = torch.zeros((), device=device)

            poss_has_target = token_mask.any(dim=(1, 2))  # [B]
            if poss_has_target.any():
                h_poss_pred_masked = h_poss_pred[poss_has_target]
                h_poss_tgt_masked = h_poss_tgt[poss_has_target]
                poss_loss = torch_functional.l1_loss(
                    h_poss_pred_masked, h_poss_tgt_masked, reduction="mean"
                )
            else:
                poss_loss = torch.zeros((), device=device)

            total_loss = token_loss + agent_loss + poss_loss

        return (
            total_loss,
            token_loss,
            agent_loss,
            poss_loss,
            H_token_pred,
            H_agent_pred,
            h_poss_pred,
        )

    def update_target_encoder(self, momentum: float):
        with torch.no_grad():
            for param_q, param_k in zip(
                self.context_encoder.parameters(), self.target_encoder.parameters()
            ):
                param_k.data.mul_(momentum).add_(param_q.data, alpha=1.0 - momentum)


**Dois ajustes de execução, sem mudar o modelo.** O gerador de máscara do lab sorteia com laços em
Python no dispositivo do tensor, o que na GPU faz centenas de sincronizações por batch. Aqui a máscara
é sorteada na CPU e enviada pronta (mesma distribuição). E a máscara ganha uma opção nova,
`ball_mask_prob`: esconde a bola num trecho contínuo da janela.


In [ ]:
_base_mask = generate_mask_vjepa


def generate_mask_vjepa(T, A, batch_size, device, mask_mode, ball_mask_prob=0.0, ball_span=0.5, **kw):
    m = _base_mask(T=T, A=A, batch_size=batch_size, device="cpu", mask_mode=mask_mode, **kw)
    if ball_mask_prob > 0:
        span = max(1, int(T * ball_span))
        pick = torch.rand(batch_size) < ball_mask_prob
        starts = torch.randint(0, T - span + 1, (batch_size,))
        for i in torch.nonzero(pick).flatten().tolist():
            m[i, starts[i]: starts[i] + span, 0] = True
    return m.to(device)


_orig_sane = FCJEPAModel._ensure_nonpad_mask_sane
FCJEPAModel._ensure_nonpad_mask_sane = staticmethod(
    lambda mask_bta, pad_bta: _orig_sane(mask_bta.cpu(), pad_bta.cpu()).to(mask_bta.device))


def new_model():
    torch.manual_seed(CFG["seed"])
    return FCJEPAModel(in_dim_enc=N_FEAT, d_model=CFG["d_model"], depth=CFG["depth"],
                       n_heads=CFG["n_heads"], same_agent_bias=True, A=A).to(DEVICE)


_m = new_model()
n_enc = sum(p.numel() for p in _m.context_encoder.parameters())
n_all = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f"encoder: {n_enc / 1e6:.2f} M parâmetros | treináveis no total: {n_all / 1e6:.2f} M")
del _m


## 5. Dados

Copia os caches do Drive para o disco da máquina (leitura bem mais rápida) e carrega tudo na memória.
Divisão:
- **treino do JEPA**: Metrica + PFF, menos os jogos de validação;
- **validação**: os `n_val_pff` primeiros jogos da PFF e os jogos em `val_metrica`, nunca vistos no
  pré-treino. A perda sai separada por provider: a da Metrica mede a generalização para um jogo novo
  de outro provider, que é a situação do jogo da competição (ele não entra no pré-treino aqui);
- **sonda**: os 3 jogos da Metrica, com rótulos H/A/D. Dois deles o encoder viu sem rótulo; o fold que
  testa no jogo de `val_metrica` (treina nos outros 2) é o **inédito**, o que imita o alvo.


In [ ]:
if CACHE_DIR != CACHE_SRC and not CACHE_DIR.exists():
    t0 = time.time()
    shutil.copytree(CACHE_SRC, CACHE_DIR)
    print(f"cópia do Drive: {time.time() - t0:.0f} s")

names = sorted(f.stem for f in CACHE_DIR.glob("*.npz") if not f.stem.startswith("target_"))
assert names, f"nenhum .npz em {CACHE_DIR}"
if CFG["max_games"]:
    names = [n for n in names if n.startswith("metrica_")] + [n for n in names if n.startswith("pff_")][: CFG["max_games"]]
GAMES = {}
for n in names:
    z = np.load(CACHE_DIR / f"{n}.npz")
    trk = Tracking(z["ball"], z["players"], z["team"])
    GAMES[n] = dict(trk=trk, vb=velocity(trk.ball, HZ), vp=velocity(trk.players, HZ),
                    pitch=tuple(float(v) for v in z["pitch"]),
                    blocks=[(int(a), int(b)) for a, b in z["blocks"]],
                    labels=z["labels"] if "labels" in z else None)

pff = [n for n in names if n.startswith("pff_")]
VAL_SETS = {"PFF": pff[: CFG["n_val_pff"]], "Metrica": [n for n in CFG["val_metrica"] if n in names]}
VAL = VAL_SETS["PFF"] + VAL_SETS["Metrica"]
TRAIN = [n for n in names if n not in VAL]
PROBE = [n for n in names if n.startswith("metrica_")]

rows = []
for n, g in GAMES.items():
    lab = g["labels"]
    inb = np.zeros(len(g["trk"].ball), bool)
    for a, b in g["blocks"]:
        inb[a: b + 1] = True
    live = (lab < 3) & inb
    rows.append(dict(jogo=n, fonte=n.split("_")[0], papel="val" if n in VAL else "treino",
                     minutos=inb.sum() / HZ / 60,
                     bola_ausente=np.isnan(g["trk"].ball[inb, 0]).mean(),
                     H=(lab[live] == 0).mean(), A=(lab[live] == 1).mean(), D=(lab[live] == 2).mean()))
SUMMARY = pd.DataFrame(rows)
display(SUMMARY.groupby(["fonte", "papel"]).agg(jogos=("jogo", "size"), minutos=("minutos", "sum"),
        bola_ausente=("bola_ausente", "mean"), H=("H", "mean"), A=("A", "mean"), D=("D", "mean")).round(3))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
for f, g in SUMMARY.groupby("fonte"):
    ax[0].hist(g["bola_ausente"], bins=20, alpha=0.6, label=f)
ax[0].set(title="bola ausente por jogo (dentro dos tempos)", xlabel="fração de amostras", ylabel="jogos")
ax[0].legend()
SUMMARY.groupby("fonte")[["H", "A", "D"]].mean().plot.bar(ax=ax[1], rot=0)
ax[1].set(title="rótulos dos eventos (média por jogo)", ylabel="fração")
plt.tight_layout(); plt.show()


In [ ]:
class Windows(Dataset):
    """Janelas [s, s + T) dentro dos blocos; rotação de 180° sorteada se flip."""

    def __init__(self, game_names, T, stride, flip, seed=0):
        self.T, self.flip, self.seed = T, flip, seed
        self.index = [(n, s) for n in game_names for a, b in GAMES[n]["blocks"]
                      for s in range(a, b - T + 2, stride)]

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        n, s = self.index[i]
        g = GAMES[n]
        tok, pad = window_tokens(g["trk"], g["vb"], g["vp"], s, self.T, g["pitch"])
        if self.flip and np.random.default_rng((self.seed, i, torch.initial_seed() % 2**31)).random() < 0.5:
            tok = flip_tokens(tok)
        return torch.from_numpy(tok), torch.from_numpy(pad)


train_ds = Windows(TRAIN, CFG["T"], CFG["stride"], flip=True, seed=CFG["seed"])
n_epoch = min(CFG["windows_per_epoch"], len(train_ds))
dl_kw = dict(batch_size=CFG["batch_size"], num_workers=CFG["num_workers"], pin_memory=DEVICE == "cuda",
             persistent_workers=CFG["num_workers"] > 0)
train_dl = DataLoader(train_ds, sampler=RandomSampler(train_ds, num_samples=n_epoch,
                      generator=torch.Generator().manual_seed(CFG["seed"])), drop_last=True, **dl_kw)
val_dls = {src: DataLoader(Windows(v, CFG["T"], CFG["stride"] * 5, flip=False), shuffle=False, **dl_kw)
           for src, v in VAL_SETS.items() if v}
STEPS_PER_EPOCH = len(train_dl)
print(f"treino: {len(TRAIN)} jogos, {len(train_ds):,} janelas possíveis, {n_epoch:,} por época "
      f"({STEPS_PER_EPOCH} batches)")
for src, dl_ in val_dls.items():
    print(f"val {src}: {VAL_SETS[src]}, {len(dl_.dataset):,} janelas")
print(f"sonda: {PROBE} | fold inédito: teste em {VAL_SETS['Metrica']}")


## 6. Métricas de acompanhamento

A perda do JEPA sozinha não diz se a representação serve para posse (no teste reduzido ela caiu e a
sonda piorou). Por isso, a cada época:

- **perda de validação** por provider (PFF e Metrica fora do pré-treino), com a mesma máscara sorteada
  em toda época. A distância entre as duas curvas é a diferença de provider;
- **saúde da representação**: rank efetivo do `h_poss` (quantas direções o vetor usa de fato; se cair
  muito, a representação colapsou) e desvio padrão médio por dimensão;
- **sonda linear H/A/D** na Metrica, leave-one-game-out (treina em 2 jogos, testa no 3º, gira), sobre
  duas leituras do mesmo encoder:
  - `h_poss`: o vetor da janela inteira;
  - `bola+times`: token da bola + média dos tokens de cada time (preserva *de que lado* está cada coisa).

  `macroF1` é a média dos 3 folds; `macroF1_inedito` é só o fold que testa no jogo fora do pré-treino.

A linha de base é o **mesmo encoder sem treino**: o pré-treino só vale se a sonda ganhar dele.


In [ ]:
@torch.no_grad()
def readouts(encoder, n, stride=CFG["probe_stride"], bs=512):
    """Leituras do encoder nas janelas de 4 s centradas em amostras H/A/D do jogo n."""
    encoder.eval()
    g, T = GAMES[n], CFG["T"]
    ks, starts = [], []
    for a, b in g["blocks"]:
        for k in range(a, b + 1, stride):
            if g["labels"][k] < 3:
                ks.append(k); starts.append(min(max(k - T // 2, a), b - T + 1))
    out = {"h_poss": [], "bola+times": []}
    for j in range(0, len(starts), bs):
        tp = [window_tokens(g["trk"], g["vb"], g["vp"], s, T, g["pitch"]) for s in starts[j: j + bs]]
        tok = torch.from_numpy(np.stack([t for t, _ in tp])).to(DEVICE)
        pad = torch.from_numpy(np.stack([p for _, p in tp])).to(DEVICE)
        _, Ha, hp = encoder(tokens=tok, attn_mask_bool=None, key_pad_bool=pad, T=T, A=A)
        out["h_poss"].append(hp.float().cpu())
        out["bola+times"].append(torch.cat([Ha[:, 0], Ha[:, 1:12].mean(1), Ha[:, 12:23].mean(1)], 1).float().cpu())
    return {k: torch.cat(v).numpy() for k, v in out.items()}, g["labels"][np.array(ks)]


def eff_rank(Z):
    s = np.linalg.svd(Z - Z.mean(0), compute_uv=False)
    p = s / s.sum()
    return float(np.exp(-(p * np.log(p + 1e-12)).sum()))


def probe(encoder):
    """Sonda LOGO na Metrica: F1 por classe e macroF1 (média dos 3 folds) por leitura, + saúde."""
    R = {n: readouts(encoder, n) for n in PROBE}
    res = {}
    for key in ("h_poss", "bola+times"):
        f1s, unseen = [], []
        for te in PROBE:
            tr = [n for n in PROBE if n != te]
            clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
            clf.fit(np.concatenate([R[n][0][key] for n in tr]), np.concatenate([R[n][1] for n in tr]))
            f1s.append(f1_score(R[te][1], clf.predict(R[te][0][key]), average=None, labels=[0, 1, 2]))
            if te in VAL_SETS["Metrica"]:
                unseen.append(f1s[-1].mean())
        f1 = np.mean(f1s, axis=0)
        res[key] = dict(f1_H=f1[0], f1_A=f1[1], f1_D=f1[2], macroF1=f1.mean(),
                        macroF1_inedito=float(np.mean(unseen)) if unseen else float("nan"))
    Z = np.concatenate([R[n][0]["h_poss"] for n in PROBE])
    res["rank"], res["std"] = eff_rank(Z), float(Z.std(0).mean())
    return res


MK = {k: v for k, v in CFG["mask"].items() if k != "mode"}


@torch.no_grad()
def val_loss(model):
    """Perda média por provider: {"PFF": [total, token, agent, poss], "Metrica": [...]}."""
    model.eval()
    res = {}
    for src, dl_ in val_dls.items():
        tot, n = np.zeros(4), 0
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(123)  # mesma máscara em toda época
            for tok, pad in dl_:
                tok, pad = tok.to(DEVICE), pad.to(DEVICE)
                keep = ~pad.all(1)
                out = model(tok[keep], pad[keep], T=CFG["T"], A=A, mask_mode=CFG["mask"]["mode"], mask_kwargs=MK)
                tot += np.array([float(x) for x in out[:4]]) * int(keep.sum()); n += int(keep.sum())
        res[src] = (tot / max(n, 1)).tolist()
    return res


t0 = time.time()
BASE = probe(new_model().context_encoder)
print(f"linha de base (encoder sem treino), {time.time() - t0:.0f} s")
display(pd.DataFrame({k: v for k, v in BASE.items() if isinstance(v, dict)}).T.round(3))
print(f"rank efetivo do h_poss: {BASE['rank']:.1f} de {CFG['d_model']} | desvio médio: {BASE['std']:.3f}")


## 7. Treino

O painel se atualiza a cada `plot_every` passos e no fim de cada época:

| painel | o que olhar |
|---|---|
| perda | treino (suavizada) deve cair; validação (pontos) deve acompanhar. Validação subindo = sobreajuste |
| componentes | token/agente/posse; se uma zera e as outras não, o modelo achou um atalho |
| lr e momento EMA | só para conferir o agendamento (warmup, cosseno; EMA subindo até 1) |
| norma do gradiente | picos grandes e frequentes = instável (baixe `lr`) |
| rank efetivo | caindo muito abaixo da linha tracejada (sem treino) = colapso |
| sonda | F1 por classe e macroF1 da melhor leitura; precisa passar da linha tracejada (sem treino) |

Checkpoint no Drive ao fim de cada época (`latest.pth`, para retomar, e `c2_encoder.pth`, para o repo).


In [ ]:
model = new_model()
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=CFG["lr"],
                        weight_decay=CFG["weight_decay"])
use_amp = CFG["amp"] and DEVICE == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
TOTAL = CFG["epochs"] * STEPS_PER_EPOCH
WARM = max(1, CFG["warmup_epochs"] * STEPS_PER_EPOCH)
HIST = dict(step=[], loss=[], token=[], agent=[], poss=[], lr=[], ema=[], gnorm=[], epochs=[])
start_epoch, step = 0, 0

latest = RUN_DIR / "latest.pth"
if latest.exists():
    ck = torch.load(latest, map_location=DEVICE, weights_only=False)
    if ck["cfg"] != CFG:
        print("AVISO: a configuração mudou desde o checkpoint; retomando mesmo assim.")
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
    HIST, step, start_epoch = ck["hist"], ck["step"], ck["epoch"] + 1
    print(f"retomando da época {start_epoch} (passo {step})")


def lr_at(s):
    if s < WARM:
        return CFG["lr"] * (s + 1) / WARM
    return CFG["lr"] * 0.5 * (1 + math.cos(math.pi * (s - WARM) / max(1, TOTAL - WARM)))


def ema_at(s):
    a, b = CFG["ema"]
    return a + (b - a) * min(1.0, s / TOTAL)


def smooth(x, k=50):
    x = np.asarray(x, float)
    if len(x) < k:
        return x
    return np.convolve(x, np.ones(k) / k, mode="valid")


def dashboard(status=""):
    clear_output(wait=True)
    fig, ax = plt.subplots(2, 3, figsize=(17, 8.5))
    s = np.array(HIST["step"]); E = HIST["epochs"]; es = [e["step"] for e in E]
    if len(s):
        k = max(1, min(50, len(s) // 5))
        ax[0, 0].plot(s[k - 1:], smooth(HIST["loss"], k), label="treino (média móvel)")
        for c in ("token", "agent", "poss"):
            ax[0, 1].plot(s[k - 1:], smooth(HIST[c], k), label=c)
        ax[0, 2].plot(s, HIST["lr"], color="C0"); a2 = ax[0, 2].twinx()
        a2.plot(s, HIST["ema"], color="C1"); a2.set_ylabel("momento EMA", color="C1")
        ax[1, 0].plot(s, HIST["gnorm"], lw=0.5, alpha=0.6); ax[1, 0].plot(s[k - 1:], smooth(HIST["gnorm"], k))
    if E:
        for src, col in (("PFF", "C1"), ("Metrica", "C3")):
            if src in E[0]["val"]:
                ax[0, 0].plot(es, [e["val"][src][0] for e in E], "o-", color=col, label=f"validação {src}")
        ax[1, 1].plot(es, [e["probe"]["rank"] for e in E], "o-", label="JEPA")
        best = [max(("h_poss", "bola+times"), key=lambda r: e["probe"][r]["macroF1"]) for e in E]
        for c, col in (("f1_H", "C0"), ("f1_A", "C1"), ("f1_D", "C2"), ("macroF1", "k"), ("macroF1_inedito", "C4")):
            ax[1, 2].plot(es, [e["probe"][b][c] for e, b in zip(E, best)], "o-", color=col, label=c)
            ax[1, 2].axhline(max(BASE[r][c] for r in ("h_poss", "bola+times")), color=col, ls="--", lw=0.8)
    ax[1, 1].axhline(BASE["rank"], color="gray", ls="--", label="sem treino")
    ax[0, 0].set(title="perda JEPA", xlabel="passo"); ax[0, 0].legend()
    ax[0, 1].set(title="componentes da perda (treino)", xlabel="passo"); ax[0, 1].legend()
    ax[0, 2].set(title="taxa de aprendizado", xlabel="passo")
    ax[1, 0].set(title="norma do gradiente (antes do corte)", xlabel="passo", yscale="log")
    ax[1, 1].set(title=f"rank efetivo do h_poss (máx. {CFG['d_model']})", xlabel="passo"); ax[1, 1].legend()
    ax[1, 2].set(title="sonda H/A/D na Metrica (LOGO) · tracejado = sem treino", xlabel="passo", ylim=(0.4, 1))
    if E:
        ax[1, 2].legend(loc="lower right", ncol=2)
    fig.suptitle(status, x=0.01, ha="left", fontsize=11)
    plt.tight_layout(); plt.show()


for ep in range(start_epoch, CFG["epochs"]):
    model.train(); t0 = time.time()
    for i, (tok, pad) in enumerate(train_dl):
        tok, pad = tok.to(DEVICE, non_blocking=True), pad.to(DEVICE, non_blocking=True)
        keep = ~pad.all(1); tok, pad = tok[keep], pad[keep]
        for gr in opt.param_groups:
            gr["lr"] = lr_at(step)
        with torch.autocast("cuda", dtype=torch.float16, enabled=use_amp):
            loss, l_tok, l_ag, l_po = model(tok, pad, T=CFG["T"], A=A, mask_mode=CFG["mask"]["mode"],
                                            mask_kwargs=MK)[:4]
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        gn = torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
        scaler.step(opt); scaler.update()
        model.update_target_encoder(ema_at(step))
        step += 1
        for key, v in (("loss", loss), ("token", l_tok), ("agent", l_ag), ("poss", l_po), ("gnorm", gn)):
            HIST[key].append(float(v.detach()))
        HIST["step"].append(step); HIST["lr"].append(lr_at(step)); HIST["ema"].append(ema_at(step))
        if not math.isfinite(HIST["loss"][-1]):
            raise RuntimeError("perda não finita: ponha amp=False ou baixe lr")
        if (i + 1) % CFG["plot_every"] == 0:
            rate = (i + 1) / (time.time() - t0)
            left = (STEPS_PER_EPOCH - i - 1) / rate + (CFG["epochs"] - ep - 1) * STEPS_PER_EPOCH / rate
            dashboard(f"época {ep + 1}/{CFG['epochs']} · passo {i + 1}/{STEPS_PER_EPOCH} · "
                      f"{rate * CFG['batch_size']:.0f} janelas/s · faltam ~{left / 60:.0f} min")
    t_train = time.time() - t0
    va, pr = val_loss(model), probe(model.context_encoder)
    HIST["epochs"].append(dict(epoch=ep, step=step, val=va, probe=pr, train_s=t_train))
    torch.save(dict(model=model.state_dict(), opt=opt.state_dict(), scaler=scaler.state_dict(),
                    hist=HIST, step=step, epoch=ep, cfg=CFG), RUN_DIR / "latest.pth")
    meta = dict(pretrain=CFG, window_frames=CFG["T"], hz=HZ, in_dim=N_FEAT, A=A, train_games=TRAIN, val_games=VAL)
    torch.save(dict(encoder=model.context_encoder.state_dict(), meta=meta, epoch=ep, val_loss=va),
               RUN_DIR / "c2_encoder.pth")
    (RUN_DIR / "history.json").write_text(json.dumps(HIST["epochs"], indent=1))
    vtxt = " · ".join(f"val {src} {v[0]:.4f}" for src, v in va.items())
    bt = pr["bola+times"]
    dashboard(f"época {ep + 1} concluída em {t_train / 60:.1f} min · {vtxt} · rank {pr['rank']:.1f} · "
              f"sonda bola+times macroF1 {bt['macroF1']:.3f} (inédito {bt['macroF1_inedito']:.3f}) · salvo")
print("treino concluído")


## 8. Resultado e exportação


In [ ]:
rows = [dict(epoca="sem treino", leitura=r, **BASE[r]) for r in ("h_poss", "bola+times")]
for e in HIST["epochs"]:
    rows += [dict(epoca=e["epoch"] + 1, leitura=r, **e["probe"][r]) for r in ("h_poss", "bola+times")]
table = pd.DataFrame(rows).round(3)
display(table)
best = table[table.epoca != "sem treino"].sort_values("macroF1").iloc[-1]
base = table[table.epoca == "sem treino"].macroF1.max()
base_u = table[table.epoca == "sem treino"].macroF1_inedito.max()
print(f"melhor sonda: época {best.epoca}, {best.leitura}, macroF1 {best.macroF1:.3f} "
      f"(sem treino: {base:.3f}) · no jogo inédito {best.macroF1_inedito:.3f} (sem treino: {base_u:.3f})")
print("encoder para o repositório:", RUN_DIR / "c2_encoder.pth")
try:
    from google.colab import files
    files.download(str(RUN_DIR / "c2_encoder.pth"))
except Exception:
    pass


## 9. Como ler o resultado

- **Sonda acima do sem treino, sobretudo em `f1_H`/`f1_A`:** o pré-treino aprendeu algo útil para
  posse. Baixe `c2_encoder.pth` para `outputs/jepa/` no repositório; lá a cabeça H/A/D + Viterbi é
  avaliada no LOGO da Metrica contra o C1 (0,592 de spellF1).
- **`macroF1_inedito` (roxo) bem abaixo do `macroF1` médio:** o ganho depende de o encoder ter visto
  o jogo. No alvo, que não entra aqui, espere o número inédito; vale então adaptar o encoder ao ATD
  localmente. A perda de validação da Metrica bem acima da da PFF aponta para o mesmo lado.
- **`bola+times` bem melhor que `h_poss`:** a informação está nos tokens, mas o pooling da posse a
  perde. Use essa leitura na cabeça.
- **Rank efetivo despencando:** colapso. Opções: subir `ema` inicial (0.998), baixar `lr`, ou
  reduzir `agent_mask_ratio`.
- **Sonda abaixo do sem treino em todas as épocas:** a tarefa de pré-treino não força a relação
  bola–time. Teste `ball_mask_prob=0.5` (a bola fica escondida em metade das janelas e o modelo
  precisa inferir onde ela está pelos jogadores), com outro `run_name`.
- Os números da sonda são por amostra, sem Viterbi e com classificador linear. Servem para comparar
  épocas e variantes entre si, não com o placar oficial.
